In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

In [7]:
df = pd.read_csv("master_features_advanced.csv")

# Define Inputs (X) and Output (y)
X = df[['Ratio_R', 'Red_Slope', 'IR_Slope', 'Red_Skew', 'IR_Skew', 'BMI']]
y = df['Glucose']

# 2. SPLIT THE DATA
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. SCALE THE DATA
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
num_predictors = X_train.shape[1]
fine_gamma = 8.0 / num_predictors
medium_gamma = 0.5 / num_predictors

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    
    "Support Vector Machine (SVR)": SVR(kernel='rbf', C=100, epsilon=0.1),
    "SVR-Fine Gaussian": SVR(kernel='rbf', gamma=fine_gamma, C=10.0, epsilon=0.1),
    "Non-Linear Medium Gaussian SVR": SVR(kernel='rbf', gamma=medium_gamma, C=10.0, epsilon=0.1),
    
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=100, random_state=42),
    
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1),
    "CatBoost Regressor": CatBoostRegressor(iterations=300, depth=4, learning_rate=0.05, verbose=0, random_seed=42)
}

In [9]:
best_rmse = float('inf')
best_model = None
best_model_name = None

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    
    print(f"\n{name}:")
    print(f"  ➡ MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.2f}")
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_model_name = name


Linear Regression:
  ➡ MAE: 20.05 | RMSE: 24.84 | R2: 0.52

Ridge Regression:
  ➡ MAE: 20.40 | RMSE: 25.28 | R2: 0.51

Lasso Regression:
  ➡ MAE: 20.18 | RMSE: 24.97 | R2: 0.52

ElasticNet:
  ➡ MAE: 20.72 | RMSE: 25.63 | R2: 0.49

K-Nearest Neighbors:
  ➡ MAE: 19.92 | RMSE: 30.46 | R2: 0.29

Support Vector Machine (SVR):
  ➡ MAE: 20.52 | RMSE: 27.73 | R2: 0.41

SVR-Fine Gaussian:
  ➡ MAE: 24.89 | RMSE: 33.78 | R2: 0.12

Non-Linear Medium Gaussian SVR:
  ➡ MAE: 21.35 | RMSE: 30.61 | R2: 0.28

Random Forest:
  ➡ MAE: 17.48 | RMSE: 22.90 | R2: 0.60

Gradient Boosting:
  ➡ MAE: 18.16 | RMSE: 22.41 | R2: 0.61

AdaBoost:
  ➡ MAE: 19.27 | RMSE: 26.63 | R2: 0.45

XGBoost:
  ➡ MAE: 11.56 | RMSE: 13.79 | R2: 0.85

LightGBM:
  ➡ MAE: 19.59 | RMSE: 26.81 | R2: 0.45


c:\Users\Revios\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



CatBoost Regressor:
  ➡ MAE: 20.08 | RMSE: 27.60 | R2: 0.41


In [10]:
joblib.dump(best_model, 'XGBoost2.joblib')
joblib.dump(scaler, 'scaler_xgb2.joblib')

print(f"\n🏆 Best Model: {best_model_name} (RMSE: {best_rmse:.2f})")
print("💾 Saved best model to 'XGBoost2.joblib'")


🏆 Best Model: XGBoost (RMSE: 13.79)
💾 Saved best model to 'XGBoost2.joblib'
